# Lección 7.6 — Interpretabilidad: importancia de variables, SHAP y dependencias parciales

> **Módulo 7 · Regularización y ajuste de hiperparámetros** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 30–40 min)
> 📋 Requisitos previos: Lección 5.4 (árboles), Lección 6.2 (Random Forest), Lección 7.1 (coeficientes y regularización), Lección 7.5 (pipelines)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Distinguir explicaciones **globales** ("¿qué importa en general?") de **locales** ("¿por qué este caso?").
- Calcular e interpretar la **importancia por permutación** y saber por qué es más fiable que `feature_importances_`.
- Leer **gráficas de dependencia parcial (PDP)** y **curvas ICE** para ver *cómo* influye cada variable.
- Explicar la idea de los **valores SHAP** (repartir una predicción entre las variables) y calcularlos en un caso pequeño.
- Evitar los errores clásicos: confundir importancia con **causa**, variables **correlacionadas** y explicar sobre el **entrenamiento**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

> 📝 La librería **SHAP** no viene en todos los entornos. En esta lección calcularemos los valores SHAP "a mano" con una función propia y, en una celda de texto, verás el código para usar la librería en Colab.

In [ ]:
import warnings
from itertools import combinations
from math import factorial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import r2_score
from sklearn.datasets import load_diabetes

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: ¿quién es el jugador más importante del equipo?

Un entrenador quiere saber qué jugador es más valioso. Podría mirar quién toca más el balón… pero eso engaña: el portero toca poco el balón y es imprescindible.

Una forma más honesta: **sacar a un jugador** (o cambiarlo por alguien al azar de la grada) y ver **cuánto empeora el equipo**. Si el equipo se hunde, ese jugador era clave. Si juega igual, no aportaba tanto.

Con los modelos hacemos lo mismo. Y nos interesan dos tipos de preguntas:

| Pregunta | Tipo | Herramienta de esta lección |
|---|---|---|
| ¿Qué variables usa más el modelo, en general? | **Global** | Importancia por permutación |
| ¿**Cómo** influye una variable: sube, baja, se satura? | **Global** | Dependencia parcial (PDP) e ICE |
| ¿Por qué el modelo le dio **este** precio a **esta** casa? | **Local** | Valores SHAP |

¿Por qué importa? Para **confiar** en el modelo (¿usa lo que debería?), **detectar errores** (¿usa una columna tramposa, Lección 2.6?), **cumplir normas** (explicar una denegación de crédito, Lección 15.6) y **aprender** del problema.

### ✋ Ejercicio 1 — Calentamiento
Clasifica cada pregunta como **global** o **local**:
1. "¿Por qué el modelo rechazó el crédito de Ana?"
2. "¿El modelo usa la edad para decidir créditos?"
3. "A partir de qué ingreso deja de importar ganar más?"
4. "¿Qué variable empujó más hacia arriba el riesgo de este paciente?"

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Local** (un caso concreto).
2. **Global** (el modelo en general).
3. **Global** (la forma del efecto de una variable: dependencia parcial).
4. **Local** (un paciente concreto: SHAP).
</details>

---
## 2. 📖 El concepto: dos formas de medir importancia

Vamos a trabajar con un **tasador de casas** ficticio. Como inventamos los datos, **sabemos la verdad** y podremos comprobar si las explicaciones aciertan:

- `m2`: superficie. **Influye mucho** (y más en zona premium).
- `distancia_centro_km`: **influye mucho**, sobre todo los primeros kilómetros.
- `antiguedad`: influye **moderadamente** (más vieja, más barata).
- `alberca`: influye **un poco**.
- `zona_premium`: sube el precio **por m²**.
- `habitaciones`: **no influye por sí misma**, pero está muy relacionada con `m2`.
- `folio`: un número de expediente **al azar**. **No influye nada**.

In [ ]:
rng = np.random.default_rng(0)
n = 2_000
casas = pd.DataFrame({
    "m2":                  rng.uniform(40, 250, n).round(),
    "distancia_centro_km": rng.uniform(0.5, 30, n).round(1),
    "antiguedad":          rng.integers(0, 60, n).astype(float),   # (las PDP piden columnas decimales)
    "alberca":             (rng.random(n) < 0.15).astype(int),
    "zona_premium":        (rng.random(n) < 0.3).astype(int),
    "folio":               rng.uniform(0, 1, n),                       # número de expediente: ruido puro
})
casas["habitaciones"] = (casas.m2 / 35 + rng.normal(0, 0.5, n)).round().clip(1, 7)

# La "verdad" (en miles de pesos). El modelo NO la conoce.
precio = (8 * casas.m2 * (1 + 0.6 * casas.zona_premium)
          + 900 * np.exp(-casas.distancia_centro_km / 8)
          - 6 * casas.antiguedad + 150 * casas.alberca
          + rng.normal(0, 150, n))

X_tr, X_te, y_tr, y_te = train_test_split(casas, precio, test_size=0.25, random_state=0)
bosque = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(f"R² en prueba del Random Forest: {bosque.score(X_te, y_te):.3f}")
casas.head()

### 📖 Método 1: la importancia "de fábrica" (`feature_importances_`)
Los modelos de árboles traen un atributo `feature_importances_` (lo usaste en la Lección 5.4). Mide **cuánto redujo el error cada variable al hacer divisiones** durante el entrenamiento. Se llama importancia por **impureza** (o MDI).

### 📖 Método 2: importancia por **permutación**
Es el "saca al jugador" de la analogía:
1. Mide el desempeño del modelo en la **prueba**.
2. **Baraja** al azar una columna (rompe su relación con el precio, pero mantiene sus valores).
3. Vuelve a medir. La **caída** del desempeño es la importancia de esa columna.
4. Repite varias veces y promedia (para no depender de un solo barajado).

### 🔮 Ejercicio 2 — ¿Dónde quedará el `folio`?
El `folio` es ruido puro. ¿En qué posición del ranking (de 7 variables) crees que lo pondrá cada método?

✍️ *Mi predicción:*

In [ ]:
imp_impureza = pd.Series(bosque.feature_importances_, index=casas.columns)
perm = permutation_importance(bosque, X_te, y_te, n_repeats=10, random_state=0, n_jobs=-1)
imp_perm = pd.Series(perm.importances_mean, index=casas.columns)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
imp_impureza.sort_values().plot.barh(ax=axes[0], color="tab:gray", title="Por impureza (feature_importances_)")
imp_perm.sort_values().plot.barh(ax=axes[1], color="tab:blue", title="Por permutación (en prueba)",
                                 xerr=pd.Series(perm.importances_std, index=casas.columns)[imp_perm.sort_values().index])
plt.tight_layout(); plt.show()

print("Posición del folio · impureza:", int(imp_impureza.rank(ascending=False)["folio"]), "de 7",
      "· permutación:", int(imp_perm.rank(ascending=False)["folio"]), "de 7")
print(f"Importancia del folio · impureza: {imp_impureza['folio']:.3f} · permutación: {imp_perm['folio']:.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Por impureza**, el `folio` recibe ≈ 0.011 y queda **6.º de 7**: ¡por delante de `alberca` (≈ 0.004), que sí influye en el precio! El ruido puro "pesa" casi tres veces más que una variable real.
- **Por permutación**, queda **último** con una importancia prácticamente **cero** (incluso un pelín negativa, por azar), mientras que `alberca` sí aparece con importancia positiva.

¿Por qué falla la impureza? El bosque, sin límites, divide hasta memorizar, y una columna continua con 2,000 valores distintos ofrece muchísimos puntos de corte para "acomodar" ruido del **entrenamiento**. La impureza premia esas divisiones aunque no sirvan con datos nuevos. La permutación se mide en la **prueba**: barajar el `folio` no cambia nada, así que su importancia es cero.

**Regla:** para decidir qué variables importan, prefiere la **importancia por permutación** sobre datos que el modelo no vio. `feature_importances_` es rápida, pero está **sesgada** hacia variables con muchos valores distintos.
</details>

---
## 3. 📖 Cómo funciona la permutación por dentro

Es tan sencilla que podemos hacerla a mano en unas pocas líneas. Barajamos **una** columna y medimos la caída del R² (Lección 4.1).

### ✏️ Ejercicio 3 — Baraja tú
Ejecuta con `COLUMNA = "m2"`. Luego ✏️ prueba `"alberca"` y `"folio"`. Compara con la gráfica azul de arriba.

In [ ]:
COLUMNA = "m2"      # ✏️ prueba "alberca", "folio", "habitaciones"

r2_original = r2_score(y_te, bosque.predict(X_te))
X_barajado = X_te.copy()
X_barajado[COLUMNA] = np.random.default_rng(0).permutation(X_barajado[COLUMNA].values)   # rompe la relación
r2_barajado = r2_score(y_te, bosque.predict(X_barajado))

print(f"R² original: {r2_original:.3f}")
print(f"R² con '{COLUMNA}' barajada: {r2_barajado:.3f}")
print(f"→ Importancia (caída del R²): {r2_original - r2_barajado:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `m2`: el R² se desploma (de ≈ 0.94 a ≈ −0.39: ¡peor que predecir siempre el promedio!). Importancia ≈ **1.3**. Es la estrella del equipo.
- `alberca`: caída muy pequeña (≈ 0.003). Aporta, pero poco: solo el 15 % de las casas tiene alberca.
- `folio`: caída ≈ **0**. Como sacar a un jugador que nunca tocaba el balón.

Los números coinciden (salvo pequeñas diferencias por el azar del barajado) con la gráfica de `permutation_importance`, que hace esto mismo 10 veces por columna y promedia.
</details>

---
## 4. 📖 ¿Qué pasa si dos variables dicen lo mismo?

La permutación tiene un punto débil: las variables **correlacionadas**. Si barajas una, el modelo puede "pedirle ayuda" a su gemela, y la caída parece pequeña. La importancia **se reparte** (o se esconde) entre las dos.

### 🔮 Ejercicio 4 — La gemela de `m2`
Añadimos una columna `m2_escrituras`: la superficie según las escrituras, casi idéntica a `m2` (con un pequeño error). Reentrenamos y recalculamos la importancia por permutación.

¿Qué crees que pasará con la importancia de `m2`?

✍️ *Mi predicción:*

In [ ]:
casas_gem = casas.assign(m2_escrituras=casas.m2 + rng.normal(0, 3, n))
Xg_tr, Xg_te = casas_gem.loc[X_tr.index], casas_gem.loc[X_te.index]
bosque_gem = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(Xg_tr, y_tr)
perm_gem = permutation_importance(bosque_gem, Xg_te, y_te, n_repeats=10, random_state=0, n_jobs=-1)

comparacion = pd.DataFrame({
    "sin gemela": imp_perm,
    "con gemela": pd.Series(perm_gem.importances_mean, index=casas_gem.columns),
}).round(3).sort_values("con gemela", ascending=False)
print(f"R² en prueba con la gemela: {bosque_gem.score(Xg_te, y_te):.3f}  (sin ella: {bosque.score(X_te, y_te):.3f})")
comparacion

<details><summary><b>👀 Ver respuesta</b></summary>

La importancia de `m2` **se desploma** (de ≈ 1.25 a ≈ 0.21), y `m2_escrituras` recibe otra parte (≈ 0.47). El modelo es igual de bueno (mismo R², 0.942), pero ahora, al barajar `m2`, puede apoyarse en su gemela.

Peligro: alguien que mire solo la tabla "con gemela" podría concluir que `m2` importa **menos** que `zona_premium` (0.21 contra 0.42), ¡y es al revés! Por eso:
- Antes de interpretar, **revisa las correlaciones** entre variables.
- Si hay grupos de variables casi iguales, interprétalos **como grupo** (o quédate con una de ellas).

¿Y `habitaciones`? Su importancia es **cero** en las dos tablas. Va de la mano con `m2`, pero no aporta nada que `m2` no diga ya, y **no influye por sí misma** en el precio: el modelo lo detectó bien.
</details>

---
## 5. 📖 Dependencia parcial: ¿*cómo* influye una variable?

La importancia dice **cuánto**, pero no **cómo**. ¿El precio sube siempre con los m²? ¿La distancia importa igual en el km 1 que en el km 25?

La **gráfica de dependencia parcial** (PDP, *partial dependence plot*) responde así:
1. Toma todas las casas.
2. Ponles a **todas** el mismo valor de la variable (p. ej. `m2 = 100`), dejando el resto como está, y promedia las predicciones.
3. Repite para `m2 = 110, 120, …` y dibuja la curva.

Es como preguntarle al modelo: *"Si todas las casas tuvieran 100 m², ¿cuánto valdrían en promedio? ¿Y con 200?"*.

### 🔮 Ejercicio 5 — Dibuja antes de ver
Recuerda la verdad: `8 × m2` (más en zona premium) y `900 × e^(−distancia/8)`. Dibuja en papel cómo crees que serán las curvas de `m2` y de `distancia_centro_km`.

✍️ *Mi dibujo / mi descripción:*

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
PartialDependenceDisplay.from_estimator(bosque, X_te, ["m2", "distancia_centro_km", "antiguedad"], ax=ax)
plt.suptitle("Dependencia parcial: precio promedio predicho (miles) según cada variable"); plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- `m2`: una **recta ascendente** (más m², más precio, de forma pareja). Coincide con la verdad lineal.
- `distancia_centro_km`: una curva que **cae rápido** en los primeros ~10 km y luego se **aplana**: vivir en el km 2 o en el km 8 cambia mucho el precio; vivir en el km 22 o en el 28, casi nada. ¡El modelo descubrió la forma exponencial sin que se la dijéramos!
- `antiguedad`: bajada suave y aproximadamente recta.

Las curvas tienen pequeños **escalones**: es la marca de los árboles (Lección 5.4), que predicen por tramos.
</details>

### 📖 ICE: una curva por casa
La PDP es un **promedio**, y los promedios esconden cosas. Las curvas **ICE** (*Individual Conditional Expectation*) dibujan **una línea por cada casa**: cómo cambiaría *su* precio al mover solo los m². La PDP es el promedio de todas esas líneas.

### ✋ Ejercicio 6 — ¿Todas las casas se comportan igual?
Ejecuta y observa las líneas finas (cada una es una casa) y la gruesa (el promedio). ¿Ves un solo "haz" de líneas o dos? ¿Qué variable crees que los separa?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
PartialDependenceDisplay.from_estimator(bosque, X_te, ["m2"], kind="both", subsample=80,
                                        centered=True, random_state=0, ax=ax)
ax.set_title("ICE: cómo cambia el precio de cada casa al mover sus m² (centrado en 0)"); plt.show()

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Hay **dos haces** de líneas con distinta pendiente: un grupo sube con más fuerza que el otro. Los separa **`zona_premium`**: en zona premium, cada m² vale un 60 % más (8 × 1.6 = 12.8 contra 8).

Esto se llama una **interacción**: el efecto de `m2` **depende** de otra variable. La PDP (línea gruesa) muestra un efecto intermedio que **no describe bien a ninguno de los dos grupos**. Cuando las ICE se abren en abanico o en grupos, desconfía del promedio.

> 💡 El parámetro `centered=True` hace que todas las líneas empiecen en 0, para comparar pendientes sin que el precio base de cada casa estorbe.
</details>

---
## 6. 📖 Valores SHAP: repartir la predicción entre las variables

Ahora la pregunta **local**: el modelo dice que **esta** casa vale unos 2,500 mil pesos. ¿Por qué? ¿Cuánto "puso" cada variable?

### La intuición: repartir la cuenta del premio
Tres amigos ganan un concurso de trivia por equipos y reciben un premio. ¿Cómo lo reparten con justicia si cada uno aportó distinto? El economista **Lloyd Shapley** (premio Nobel 2012) propuso: imagina que los jugadores llegan **en todos los órdenes posibles**, mide cuánto **suma** cada uno al llegar, y promedia. Eso es su **valor de Shapley**.

Con un modelo:
- Los **jugadores** son las variables de la casa.
- El **premio** es la diferencia entre la predicción para **esta** casa y la predicción **promedio** (la "casa típica").
- El valor SHAP de cada variable es su parte justa de esa diferencia.

```
precio predicho  =  precio promedio  +  SHAP(m2)  +  SHAP(distancia)  +  ...  +  SHAP(folio)
     2,300       =      1,350        +   +600     +     +350          +  ...  +     0      (números de ejemplo)
```

Propiedad clave: **las contribuciones suman exactamente** la diferencia. Nada se pierde ni se inventa.

### 🔮 Ejercicio 7 — Una casa concreta
Elegimos una casa grande, en zona premium y cerca del centro. Antes de calcular: ¿qué variables crees que tendrán SHAP **positivo** (suben su precio respecto al promedio) y cuáles **negativo**? ¿Cuál será el SHAP del `folio`?

In [ ]:
casa = pd.Series({"m2": 180, "distancia_centro_km": 3.0, "antiguedad": 40, "alberca": 0,
                  "zona_premium": 1, "folio": 0.5, "habitaciones": 5})[casas.columns]
print(casa.to_string())
print(f"\nPrecio predicho para esta casa: {bosque.predict(casa.to_frame().T)[0]:,.0f} mil")

✍️ *Mi predicción:*

La función `shap_casero` calcula los valores SHAP **exactos** siguiendo la idea de Shapley: prueba todos los subconjuntos de variables "conocidas" (las demás se rellenan con casas de fondo al azar) y promedia la aportación de cada variable. No necesitas entender cada línea; fíjate en el resultado.

> ⏱️ Con 7 variables son 2⁷ = 128 subconjuntos: tarda unos segundos. La librería SHAP usa atajos muy ingeniosos para hacerlo rápido con modelos de árboles.

In [ ]:
def shap_casero(modelo, x, fondo):
    # Valores SHAP exactos: "precio medio si solo conociéramos las variables de S"
    columnas, n = list(x.index), len(x)
    memoria = {}
    def valor(S):
        S = frozenset(S)
        if S not in memoria:
            datos = fondo.copy()
            for c in S:
                datos[c] = x[c]                       # fijamos las variables "conocidas" a las de esta casa
            memoria[S] = modelo.predict(datos).mean()
        return memoria[S]
    shap = {}
    for c in columnas:
        otros = [o for o in columnas if o != c]
        shap[c] = sum(factorial(k) * factorial(n - k - 1) / factorial(n) * (valor(S + (c,)) - valor(S))
                      for k in range(n) for S in combinations(otros, k))
    return pd.Series(shap), valor(())

fondo = X_tr.sample(150, random_state=0)                 # "casas típicas" para comparar
valores_shap, base = shap_casero(bosque, casa, fondo)

print(f"Precio promedio (base): {base:,.0f}")
print(f"Base + suma de SHAP   : {base + valores_shap.sum():,.0f}")
print(f"Predicción del modelo : {bosque.predict(casa.to_frame().T)[0]:,.0f}   ← ¡coinciden!\n")
print(valores_shap.sort_values(ascending=False).round(0).to_string())

orden = valores_shap.sort_values()
plt.barh(orden.index, orden.values, color=["tab:red" if v < 0 else "tab:green" for v in orden])
plt.axvline(0, color="k", lw=.8); plt.xlabel("Contribución al precio (miles)")
plt.title("¿Por qué esta casa vale lo que vale? (valores SHAP)"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La casa vale ≈ 2,554 contra ≈ 1,356 de la casa típica: hay ≈ 1,200 de diferencia que repartir.
- **Positivos (verde):** `zona_premium` (≈ +536), `m2` (≈ +382, es grande) y `distancia_centro_km` (≈ +322, está cerca del centro).
- **Negativo (rojo):** `antiguedad` (40 años) le resta ≈ 34. `alberca` (no tiene) resta un poquito.
- `folio` y `habitaciones`: **casi 0** (≈ ±8, el pequeño ruido del bosque). Correcto.

¿Por qué `zona_premium` suma más que `m2`? Porque en la verdad, la zona premium **multiplica** el precio por m², y esta casa es grande: el "premio" de la interacción se reparte entre las dos variables, y SHAP le asigna una buena parte a la zona.

Y la propiedad mágica: **precio promedio + suma de los SHAP = predicción exacta** (1,356 + 1,198 = 2,554). Es una explicación que "cuadra la cuenta".

💬 Explicación para el cliente: *"Su casa vale bastante más que una casa típica sobre todo por estar en zona premium, por su tamaño y por estar cerca del centro; su antigüedad le resta algo de valor"*.
</details>

### 📖 SHAP con la librería oficial (para Colab)
En la práctica nadie calcula SHAP a mano: la librería **`shap`** lo hace en milisegundos para modelos de árboles y ofrece gráficas muy útiles. Así se usa:

```python
# ⚠️ Para Colab: la librería SHAP no viene en este entorno. En Colab, primero ejecuta:  !pip install shap
import shap

explicador = shap.TreeExplainer(bosque)           # atajo rápido y exacto para modelos de árboles
valores = explicador(X_te)                        # valores SHAP de todas las casas de prueba

shap.plots.waterfall(valores[0])                  # LOCAL: la "cascada" de una casa (como nuestra gráfica de barras)
shap.plots.beeswarm(valores)                      # GLOBAL: un punto por casa y variable; color = valor de la variable
shap.plots.scatter(valores[:, "m2"], color=valores[:, "zona_premium"])   # como una PDP, mostrando la interacción
```

> 💡 El **beeswarm** junta lo mejor de las dos visiones: el orden de las variables es una importancia global (el promedio del tamaño de sus SHAP), y cada punto es una explicación local.

---
## 7. 📖 Errores comunes y señales de alerta

### 🔮 Ejercicio 8 — Explicar sobre el entrenamiento
Calculamos la importancia por permutación del `folio` dos veces: sobre la **prueba** (como hasta ahora) y sobre el **entrenamiento**. ¿Crees que darán lo mismo?

✍️ *Mi predicción:*

In [ ]:
for nombre, Xp, yp in [("entrenamiento", X_tr, y_tr), ("prueba", X_te, y_te)]:
    p = permutation_importance(bosque, Xp, yp, n_repeats=10, random_state=0, n_jobs=-1)
    imp = pd.Series(p.importances_mean, index=casas.columns)
    print(f"Sobre {nombre:<13}: importancia del folio = {imp['folio']:.4f}  ·  de alberca = {imp['alberca']:.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No dan lo mismo.** Sobre el **entrenamiento**, el `folio` tiene una importancia **positiva** (≈ 0.0095), ¡casi el doble que `alberca` (≈ 0.005)!, porque el bosque lo usó para memorizar ruido en esas mismas filas. Sobre la **prueba**, el `folio` vale ≈ **0** y `alberca`, que sí es real, conserva su importancia (≈ 0.003).

Explicar sobre el entrenamiento te cuenta **lo que el modelo memorizó**, no lo que **generaliza**. Usa datos que el modelo no vio.
</details>

| 🚨 Error | Por qué es un problema | Qué hacer |
|---|---|---|
| "La variable X es la más importante, así que **causa** el resultado" | Importancia = lo que **el modelo** usa para predecir, no lo que **causa** el resultado en el mundo real | Hablar de "el modelo se apoya en X". Para causas se necesitan experimentos o métodos causales |
| Confiar en `feature_importances_` (impureza) | Sesgada hacia variables con muchos valores; se mide en el entrenamiento | Importancia por permutación sobre datos no vistos |
| Interpretar variables correlacionadas por separado | La importancia se reparte o se esconde (Sección 4) | Revisar correlaciones; interpretar en grupo |
| Explicar sobre el entrenamiento | Muestra lo memorizado (Ejercicio 8) | Usar la prueba o la validación |
| Leer una PDP en zonas sin datos | Con variables correlacionadas, la PDP inventa casas imposibles (p. ej. 40 m² con 7 habitaciones) | Mirar también la distribución de los datos (las "rayitas" de la gráfica) |
| Fiarse solo de la PDP | El promedio esconde interacciones | Mirar las ICE (Sección 5) |
| Explicar un modelo malo | Explicas con detalle… algo que no funciona | Primero, un buen desempeño en prueba; luego, explicar |

### ✋ Ejercicio 9 — ¿Causa o predicción?
Un modelo de hospital predice el riesgo de complicaciones, y la variable más importante es "**recibió el medicamento X**": los que lo recibieron tienen **más** riesgo. Un directivo propone dejar de dar el medicamento X. ¿Qué le dirías?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Que el modelo **no dice que el medicamento cause complicaciones**. Lo más probable es lo contrario: los médicos dan el medicamento X a los pacientes **más graves**, así que "recibió X" es una **señal** de gravedad. El modelo lo usa porque **predice** bien, no porque sea una **causa**. Retirarlo podría ser peligroso. Para saber el efecto real del medicamento se necesita un ensayo controlado o un análisis causal específico.
</details>

---
## 8. 📋 Ficha de las técnicas

| Técnica | Pregunta que responde | Tipo | ✅ Fortalezas | ❌ Debilidades | En scikit-learn |
|---|---|---|---|---|---|
| Coeficientes (lineales) | ¿Cuánto cambia la predicción por unidad? | Global | Exactos, simples | Solo modelos lineales; exigen escalar para comparar (Lección 7.1) | `.coef_` |
| Impureza (MDI) | ¿Qué variables usó el árbol al dividir? | Global | Gratis, rápida | Sesgada; mide sobre el entrenamiento | `.feature_importances_` |
| Permutación | ¿Cuánto empeora el modelo sin esta variable? | Global | Cualquier modelo; sobre datos no vistos | Sufre con variables correlacionadas; cuesta varios cálculos | `permutation_importance` |
| PDP / ICE | ¿**Cómo** cambia la predicción con esta variable? | Global (PDP) / por caso (ICE) | Muestra la forma del efecto e interacciones (ICE) | Inventa combinaciones imposibles si hay correlación | `PartialDependenceDisplay` |
| SHAP | ¿Cuánto aportó cada variable a **esta** predicción? | Local (y global agregado) | Reparto justo que cuadra la cuenta; muy expresivo | Cálculo costoso (salvo árboles); también sufre con correlaciones | Librería `shap` (Colab) |

## 9. 🧠 Resumen en 5 puntos
1. Hay preguntas **globales** (¿qué usa el modelo en general? ¿cómo influye cada variable?) y **locales** (¿por qué este caso?).
2. La **importancia por permutación** baraja una columna y mide la caída del desempeño en datos **no vistos**; es más fiable que `feature_importances_`.
3. La **dependencia parcial (PDP)** muestra la **forma** del efecto; las **ICE** revelan si distintos casos se comportan distinto (interacciones).
4. Los **valores SHAP** reparten la predicción de un caso entre sus variables, y **suman exactamente** la diferencia con la predicción promedio.
5. Importancia **no es causa**; las variables **correlacionadas** confunden a todas las técnicas; explica siempre sobre datos que el modelo no vio.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30–40 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** La importancia por permutación de una variable es **negativa** (muy cerca de cero). ¿Qué significa?
- a) Es la variable más importante · b) El modelo no la usa de forma útil; barajarla da igual (o mejora por azar) · c) Hay un error · d) Causa el resultado al revés

**2.** Quieres explicarle a **un cliente** por qué su préstamo fue rechazado. ¿Qué técnica usas?
- a) `feature_importances_` · b) Una PDP · c) Valores SHAP de ese cliente · d) La exactitud del modelo

**3.** En las curvas ICE de `edad`, la mitad de las líneas suben y la otra mitad bajan. La PDP sale **plana**. ¿Qué concluyes?
- a) La edad no influye · b) Hay una interacción: la edad influye en sentidos opuestos según el grupo · c) El modelo está mal · d) Hay que quitar la edad

**4.** Dos variables casi idénticas tienen cada una una importancia por permutación pequeña. ¿Qué haces?
- a) Quitar las dos · b) Barajarlas **juntas** (o quitar una) y volver a medir · c) Concluir que no importan · d) Usar `feature_importances_`

**5.** La suma de los valores SHAP de una predicción es igual a…
- a) La predicción · b) La predicción menos la predicción promedio · c) 1 · d) El R² del modelo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Barajarla no empeora el modelo; una ligera mejora es pura casualidad del barajado.
2. **c)** Es una pregunta **local**: SHAP de ese cliente.
3. **b)** Los efectos opuestos se cancelan en el promedio; la PDP sola te habría engañado.
4. **b)** Medidas por separado, cada una "esconde" a la otra (Sección 4).
5. **b)** Base (promedio) + suma de SHAP = predicción.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Un banco usa un modelo con la variable `codigo_postal`. La importancia por permutación dice que es la 2.ª más importante. ¿Qué riesgo ves? (Pista: Lección 15.4 y el "proxy" del proyecto del Módulo 5).

**B. Caso de análisis 💼** Tu modelo de rotación de empleados dice que la variable más importante es `horas_extra`. Recursos Humanos quiere prohibir las horas extra para reducir la rotación. ¿Qué les recomendarías antes de tomar esa decisión?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** El código postal puede funcionar como **proxy** de origen étnico o nivel socioeconómico: el modelo podría estar discriminando sin usar explícitamente esas variables. La importancia alta es una **señal de alerta** para auditar: medir los errores y las tasas de aprobación por grupo, y valorar si es legal y ético usarla.

**B.** Que la importancia muestra **asociación**, no causa: quizá los empleados que hacen horas extra están en equipos con mala gestión o falta de personal, y eso es lo que provoca la rotación. Recomendaciones: mirar la PDP/ICE (¿el riesgo sube a partir de cierto número de horas? ¿en todos los departamentos?), explorar SHAP de casos concretos, y si es posible, probar un cambio **controlado** (un piloto en algunos equipos) antes de una política general.
</details>

### 10.3 🧩 Práctica integradora: el informe de interpretabilidad para la clínica

La clínica de la Lección 7.1 (diabetes, 10 variables) tiene un modelo de **gradient boosting** y te pide un informe: *qué variables usa, cómo influyen y por qué le asignó un riesgo alto a un paciente concreto*.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Entrena y comprueba que el modelo vale la pena**

In [ ]:
X_d, y_d = load_diabetes(return_X_y=True, as_frame=True)
X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(X_d, y_d, test_size=0.25, random_state=0)

modelo_d = HistGradientBoostingRegressor(max_iter=100, learning_rate=0.05, max_depth=2,
                                         min_samples_leaf=30, random_state=0).fit(X_d_tr, y_d_tr)
print(f"R² en prueba: {modelo_d.score(X_d_te, y_d_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

R² en prueba ≈ **0.30**. Modesto (recuerda que este conjunto de prueba es exigente, Lección 7.1), pero el modelo aprende algo real. Lo comprobamos **antes** de explicar: explicar un modelo que no funciona no tiene sentido.
</details>

**Paso 2 🔮 — ¿Qué variables usa?** Antes de ejecutar: en la Lección 7.1, Lasso se quedó con `bmi`, `s5` y `bp` como las de mayor peso. ¿Crees que el boosting coincidirá?

✍️ *Mi apuesta:*

In [ ]:
perm_d = permutation_importance(modelo_d, X_d_te, y_d_te, n_repeats=20, random_state=0, n_jobs=-1)
imp_d = pd.DataFrame({"importancia": perm_d.importances_mean, "desv": perm_d.importances_std},
                     index=X_d.columns).sort_values("importancia", ascending=False)
imp_d.round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

Sí, en lo esencial: **`s5`** (≈ 0.16) y **`bmi`** (≈ 0.14) encabezan la lista con mucha diferencia, y `bp` aparece tercera (≈ 0.035). Dos modelos muy distintos (uno lineal con L1 y uno de árboles) coinciden en las variables clave: eso **aumenta la confianza** en la conclusión. Varias variables tienen importancias cercanas a cero o con una desviación tan grande como la propia importancia: no podemos afirmar que el modelo las use de forma útil.
</details>

**Paso 3 ✋ — ¿Cómo influyen las dos principales?** Dibuja la PDP con ICE de las dos variables más importantes y descríbelas en palabras.

In [ ]:
top2 = list(imp_d.index[:2])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
PartialDependenceDisplay.from_estimator(modelo_d, X_d_te, top2, kind="both", subsample=60, random_state=0, ax=ax)
plt.tight_layout(); plt.show()

✍️ *Mi descripción:*

<details><summary><b>👀 Ver respuesta</b></summary>

Para `s5` y `bmi` (índice de masa corporal), la línea promedio **sube**: a mayor valor, mayor progresión de la enfermedad según el modelo. La subida no es una recta perfecta: tiene tramos más planos y otros más empinados (escalones típicos de los árboles), y en los extremos se aplana porque hay pocos pacientes. Las líneas ICE van casi **paralelas**: no se ven interacciones fuertes, el efecto es parecido para todos los pacientes.

> 📝 Los valores de este conjunto de datos vienen **estandarizados** (centrados en 0), por eso el eje no está en kg/m². En un informe real, habría que convertirlos a unidades clínicas.
</details>

**Paso 4 ▶️ — El paciente con el riesgo predicho más alto.** Calculamos sus valores SHAP con nuestra función (10 variables → 1,024 subconjuntos; tarda unos segundos).

In [ ]:
predicciones = pd.Series(modelo_d.predict(X_d_te), index=X_d_te.index)
paciente = X_d_te.loc[predicciones.idxmax()]
shap_pac, base_pac = shap_casero(modelo_d, paciente, X_d_tr.sample(80, random_state=0))

print(f"Riesgo promedio (base): {base_pac:.0f} · Predicción para este paciente: {predicciones.max():.0f}")
print(f"Comprobación · base + suma SHAP = {base_pac + shap_pac.sum():.0f}\n")
print("Contribuciones (de mayor a menor efecto):")
print(shap_pac.reindex(shap_pac.abs().sort_values(ascending=False).index).round(1).to_string())

✋ ¿Qué dos variables explican la mayor parte de su riesgo alto? ¿Hay alguna que lo **reduzca**?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

La predicción de este paciente está muy por encima del promedio (≈ 286 contra ≈ 162). De esos ≈ 124 puntos de diferencia, **`bmi`** explica ≈ 56 y **`s5`** ≈ 30: tiene valores altos en las dos. Le sigue `bp` (≈ 16). La comprobación cuadra: base + suma de SHAP = predicción.

La única variable con contribución negativa apreciable es `sex` (≈ −2.5): le **resta** un poquito de riesgo. Es tan pequeña que no cambia la conclusión, pero un buen informe menciona también lo que juega a favor del paciente.
</details>

**Paso 5 ✋ — El informe.** Escribe 4–5 frases para el director médico, sin jerga, que incluyan: qué tan bueno es el modelo, qué variables usa, cómo influyen, el caso del paciente, y **una advertencia** sobre lo que el modelo **no** puede decir.

✍️ *Mi informe:*

<details><summary><b>👀 Ver un ejemplo</b></summary>

*"El modelo predice la progresión de la diabetes a un año con una precisión moderada: ayuda a ordenar a los pacientes por riesgo, pero no es exacto caso por caso. Se apoya sobre todo en dos mediciones: el índice de masa corporal y el nivel del marcador s5 en sangre, y en menor medida en la presión arterial. Para ambas, a valores más altos, mayor progresión prevista, de forma parecida en todos los pacientes. El paciente con mayor riesgo previsto lo es principalmente por tener los dos valores muy elevados. Advertencia: el modelo detecta asociaciones; no demuestra que bajar esos valores reduzca la progresión, aunque sea coherente con lo que sabemos clínicamente. Esa decisión debe apoyarse en evidencia médica."*
</details>

---
## 📚 Para profundizar
- 📖 **Libro (gratis en línea):** Christoph Molnar, *Interpretable Machine Learning*: capítulos de permutación, PDP, ICE y SHAP.
- 📘 **Documentación:** scikit-learn, *Inspection* (`permutation_importance`, `PartialDependenceDisplay`) y el ejemplo *Permutation Importance vs Random Forest Feature Importance (MDI)*.
- 🧪 **SHAP:** documentación oficial de la librería `shap` (shap.readthedocs.io), con ejemplos listos para Colab.

---
⬅️ **Anterior:** [7.5 Pipelines: evitar fugas mientras se ajusta](7.5-pipelines-sin-fugas.ipynb) · ➡️ **Siguiente:** [7.7 Proyecto del módulo: Afinar con presupuesto limitado: las casas de Ames 🎛️](7.7-proyecto-del-modulo.md)